# wrap-forward-fn-generic — ex1: write the wrap_forward_fn shell — unbox, call, box

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `wrap-forward-fn-generic`. Running the final beacon cell reports progress against the `Backprop: wrap forward fn` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: wrap forward fn` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`wrap-forward-fn-generic`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "wrap-forward-fn-generic"
DD_SUBTOPIC = "Backprop: wrap forward fn"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## wrap_forward_fn — quick refresher

`wrap_forward_fn` is the **factory** that turns a plain numerical fn (`torch.log`, `torch.multiply`, ...) into an autograd-aware version that (a) unboxes Tensor → raw, (b) runs the forward, (c) boxes the result back into a Tensor, and (d) attaches a Recipe so the reverse pass can replay the call.

```python
def wrap_forward_fn(fwd_fn):
    def tensor_func(*args, **kwargs):
        raw_args = tuple(a.array if isinstance(a, Tensor) else a for a in args)
        out_raw = fwd_fn(*raw_args, **kwargs)
        out = Tensor(out_raw)
        # if any input is a tracked Tensor, attach a Recipe
        out.recipe = Recipe(fwd_fn, raw_args, kwargs, parents)
        return out
    return tensor_func
```

The closure captures `fwd_fn` — one factory call replaces dozens of hand-written wrappers. Every wrapper shares the same unbox-call-box skeleton; only `fwd_fn` varies.

### Exercise 1 — write the wrap_forward_fn shell — unbox, call, box

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Create
> LO: Create the wrap_forward_fn factory closure that converts a raw numerical fn into a Tensor-aware wrapper via the unbox → call → box pattern.
> Keywords: closure, factory, unbox, box, tensor-wrapper
> ```

**KCs targeted:** `wrap-forward-fn-generic`, `unbox-args-tensor-to-array`

You are given a minimal `Tensor` wrapper class (a thin shell over a raw `torch.Tensor` stored on `.array`).

Implement `wrap_forward_fn(fwd_fn)`. It must return a NEW function `tensor_func(*args, **kwargs)` that:

1. **Unbox** — for each arg, if it is a `Tensor` instance, replace it    with `.array`; otherwise pass it through unchanged. (Plain ints /    floats / raw torch tensors should pass through.)
2. **Call** — invoke `fwd_fn(*raw_args, **kwargs)`.
3. **Box** — wrap the result in a new `Tensor(out_raw)` and return it.

Do NOT touch Recipe / autograd in this drill — pure unbox/call/box. (Recipe attachment is the next drill.)

The point: `wrap_forward_fn` is a closure factory. ONE definition handles every op — `log`, `exp`, `multiply`, `sum` — because the only thing that varies is the captured `fwd_fn`. Without this factory, ARENA would need 20+ hand-written wrappers.

In [ ]:
from dataclasses import dataclass


class Tensor:
    """Tiny wrapper around a raw torch.Tensor stored on .array."""
    def __init__(self, array):
        # Coerce to torch.Tensor if a list / number snuck in.
        self.array = array if isinstance(array, t.Tensor) else t.tensor(array)
    def __repr__(self):
        return f'Tensor({self.array.tolist()})'


def wrap_forward_fn(fwd_fn):
    """Return a Tensor-aware wrapper that unboxes args, calls fwd_fn, boxes the result."""
    raise NotImplementedError()


def _test_ex1():
    # Wrap torch.log — unary op on Tensor.
    tlog = wrap_forward_fn(t.log)
    a = Tensor(t.tensor([1.0, t.e, t.e ** 2]))
    b = tlog(a)
    assert isinstance(b, Tensor), f'wrap must return a Tensor, got {type(b)}'
    assert isinstance(b.array, t.Tensor), '.array must be a torch.Tensor'
    expected = t.tensor([0.0, 1.0, 2.0])
    assert t.allclose(b.array, expected, atol=1e-5), f'log fail: {b.array}'

    # Wrap torch.multiply — binary op. Both args wrapped as Tensor.
    tmul = wrap_forward_fn(t.multiply)
    x = Tensor(t.tensor([2.0, 3.0]))
    y = Tensor(t.tensor([5.0, 7.0]))
    z = tmul(x, y)
    assert isinstance(z, Tensor)
    assert t.allclose(z.array, t.tensor([10.0, 21.0]))

    # Mixed: one Tensor arg, one raw scalar — scalar must pass through unchanged.
    z2 = tmul(x, 2.0)  # 2.0 is NOT a Tensor; should pass through to torch.multiply
    assert isinstance(z2, Tensor)
    assert t.allclose(z2.array, t.tensor([4.0, 6.0])), f'mixed scalar fail: {z2.array}'

    # kwargs must thread through to fwd_fn.
    tsum = wrap_forward_fn(t.sum)
    m = Tensor(t.tensor([[1.0, 2.0], [3.0, 4.0]]))
    row_sums = tsum(m, dim=1)
    assert t.allclose(row_sums.array, t.tensor([3.0, 7.0])), f'kwargs fail: {row_sums.array}'

    # wrap_forward_fn must produce a DIFFERENT callable each call (a new closure),
    # and the captured fwd_fn must NOT leak between wrapped fns.
    tneg = wrap_forward_fn(t.neg)
    assert tneg is not tlog, 'each wrap must return a fresh closure'
    n = tneg(Tensor(t.tensor([1.0, -2.0])))
    assert t.allclose(n.array, t.tensor([-1.0, 2.0])), 'neg wrapper fail'
    # tlog still works after wrapping tneg.
    assert t.allclose(tlog(Tensor(t.tensor([1.0]))).array, t.tensor([0.0]))
    _dd_passed.add('ex1')
    print("ex1 ✓")

_test_ex1()

<details><summary>Solution</summary>

```python
from dataclasses import dataclass


class Tensor:
    def __init__(self, array):
        self.array = array if isinstance(array, t.Tensor) else t.tensor(array)
    def __repr__(self):
        return f'Tensor({self.array.tolist()})'


def wrap_forward_fn(fwd_fn):
    def tensor_func(*args, **kwargs):
        # 1. Unbox: Tensor → .array; anything else passes through.
        raw_args = tuple(a.array if isinstance(a, Tensor) else a for a in args)
        # 2. Call the wrapped fn on raw tensors.
        out_raw = fwd_fn(*raw_args, **kwargs)
        # 3. Box the result back into a Tensor.
        return Tensor(out_raw)
    return tensor_func
```

**Why a closure beats a class.** Each wrapped op gets its own `tensor_func` closure that captures `fwd_fn`. No shared mutable state. The factory pattern is dense — three lines of body produce N callables.

**Why `isinstance(a, Tensor) else a`.** Real call sites mix wrapped Tensors and raw scalars (`a * 2.0`, `x.sum(dim=1)`). The unbox map must be a no-op on non-Tensor args so the wrapped fn receives them unchanged. The dual side (boxing scalars TO Tensor) is the `__rmul__` problem and lives elsewhere.

**This drill is intentionally Recipe-free.** Adding the Recipe is one extra block — see the next exercise. The mental model is cleaner if you nail the unbox/call/box mechanism first.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()